# EarthCARE Aerosol–Cloud Interactions: Three-Region Analysis

**Branch:** `methodology-improvements`

This notebook loads pre-processed data for three climatically distinct regions, runs K-means clustering with consistent methodology, and produces comparative figures for the final report.

**Regions:**
- West Pacific (WP): 100–160°E, 0–20°N — anthropogenic + biomass burn aerosols
- East Pacific (EP): 160–100°W, 0–20°N — predominantly sea salt aerosols  
- Antarctica / Ross Sea (AN): 160–220°E (0–360 convention), 80–60°S — pristine environment

**Prerequisites:** run `merger.ipynb`, `merger_EP.ipynb`, and the Antarctica merger to produce the `.nc` input files.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans

## Helper functions

In [ ]:
def prepare_region(nc_numerical, nc_complete, bbox_fn, k=5, random_state=42):
    """Load, filter, normalise, cluster, and return all artefacts for one region.

    Parameters
    ----------
    nc_numerical : str
        Path to challenge_1min_numerical_*.nc
    nc_complete : str
        Path to challenge_1min_complete_*.nc (includes stc_* labels)
    bbox_fn : callable
        Function that takes an xarray.Dataset and returns a boolean DataArray
        for the geographic + ocean filter (land_flag < 0.1 is applied inside).
    k : int
        Number of K-means clusters.
    random_state : int
        Seed for KMeans reproducibility.

    Returns
    -------
    dict with keys:
        ds           : filtered xarray.Dataset (numerical, with lat/lon)
        x            : ndarray (n_samples, 3) — log1p + min-max normalised IWP/LWP/AOT
        y_kmeans     : ndarray (n_samples,) — cluster labels 0..k-1
        kmeans_model : fitted KMeans instance
        ds_tc        : filtered + time-aligned xarray.Dataset with stc_* labels
    """
    # --- numerical data ---
    ds = xr.open_dataset(nc_numerical)
    ds = ds.where(bbox_fn(ds) & (ds.land_flag < 0.1), drop=True)

    x = ds.drop_vars(['latitude', 'longitude', 'land_flag']).to_array().transpose('time', 'variable').values
    x = np.log1p(x)
    x = (x - np.min(x, axis=0)) / (np.max(x, axis=0) - np.min(x, axis=0))

    kmeans_model = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    y_kmeans = kmeans_model.fit_predict(x)

    # --- synergetic labels, aligned to the same time index ---
    ds_tc = xr.open_dataset(nc_complete)
    ds_tc = ds_tc.where(bbox_fn(ds_tc) & (ds_tc.land_flag < 0.1), drop=True)
    ds_tc = ds_tc.sel(time=ds.time)

    return dict(ds=ds, x=x, y_kmeans=y_kmeans, kmeans_model=kmeans_model, ds_tc=ds_tc)

In [ ]:
def bbox_wp(ds):
    return (ds.latitude > 0) & (ds.latitude < 20) & (ds.longitude > 100) & (ds.longitude < 160)

def bbox_ep(ds):
    return (ds.latitude > 0) & (ds.latitude < 20) & (ds.longitude > -160) & (ds.longitude < -100)

def bbox_an(ds):
    lon360 = ds.longitude % 360
    return (ds.latitude > -80) & (ds.latitude < -60) & (lon360 > 160) & (lon360 < 220)

In [ ]:
# Aggregate the 36 stc classes into 6 physically meaningful macro-groups
STC_GROUPS = {
    'Clear sky':     [1],
    'Precipitation': [5, 6, 7],
    'Liquid cloud':  [8, 9, 10, 11],
    'Ice cloud':     list(range(13, 22)),
    'Aerosol':       list(range(26, 35)),
    'Other':         [-1, 0, 2, 3, 4, 12, 22, 23, 24, 25],
}
GROUP_COLORS = {
    'Clear sky':     '#a2cffe',
    'Precipitation': '#042e60',
    'Liquid cloud':  '#f5bf03',
    'Ice cloud':     '#0d75f8',
    'Aerosol':       '#7a9703',
    'Other':         '#c5c9c7',
}

def stc_to_group(val):
    """Map a single stc integer value to its macro-group name."""
    for group, members in STC_GROUPS.items():
        if val in members:
            return group
    return 'Other'

def stc_composition(ds_tc, stc_var, y_kmeans, k):
    """Return a (k x n_groups) array of percentage composition for one stc level.

    Parameters
    ----------
    ds_tc   : xr.Dataset with stc_* variables
    stc_var : str — e.g. 'stc_10000'
    y_kmeans: ndarray (n_samples,)
    k       : int

    Returns
    -------
    ndarray (k, n_groups), group_names list
    """
    group_names = list(STC_GROUPS.keys())
    result = np.zeros((k, len(group_names)))
    vals = ds_tc[stc_var].values
    for c in range(k):
        subset = vals[y_kmeans == c]
        total = len(subset)
        if total == 0:
            continue
        for j, group in enumerate(group_names):
            result[c, j] = np.isin(subset, STC_GROUPS[group]).sum() / total * 100
    return result, group_names

## Run clustering for all three regions

In [ ]:
K = 5  # number of clusters — revisit after checking elbow plots per region

regions = {
    'West Pacific': prepare_region(
        'challenge_1min_numerical.nc', 'challenge_1min_complete.nc', bbox_wp, k=K),
    'East Pacific': prepare_region(
        'challenge_1min_numerical_EP.nc', 'challenge_1min_complete_EP.nc', bbox_ep, k=K),
    'Antarctica': prepare_region(
        'challenge_1min_numerical_AN.nc', 'challenge_1min_complete_AN.nc', bbox_an, k=K),
}

region_names = list(regions.keys())
print({r: len(regions[r]['y_kmeans']) for r in region_names})

## Figure 1 — Feature space: AOT / IWP / LWP per region

Three rows (one per region) × three columns (AOT vs IWP, AOT vs LWP, IWP vs LWP).  
All axes share the same [0, 1] normalised scale so cluster geometry is directly comparable across regions.

In [ ]:
# Column indices in the normalised matrix x: 0=IWP, 1=LWP, 2=AOT
feature_cols = {'iwp': 0, 'lwp': 1, 'aot': 2}
pairs = [('aot', 'iwp'), ('aot', 'lwp'), ('iwp', 'lwp')]

fig, axes = plt.subplots(3, 3, figsize=(15, 13), sharex='col', sharey='col')
fig.suptitle('K-means clusters in feature space (log1p + min-max normalised)', fontsize=14, y=1.01)

for row, region in enumerate(region_names):
    r = regions[region]
    x, y_kmeans, km = r['x'], r['y_kmeans'], r['kmeans_model']

    for col, (vx, vy) in enumerate(pairs):
        ax = axes[row, col]
        ax.scatter(x[:, feature_cols[vx]], x[:, feature_cols[vy]],
                   c=y_kmeans, s=6, cmap='tab10', vmin=0, vmax=9, alpha=0.5)
        centers = km.cluster_centers_
        ax.scatter(centers[:, feature_cols[vx]], centers[:, feature_cols[vy]],
                   c='red', s=60, marker='X', zorder=5)
        for i in range(len(centers)):
            ax.text(centers[i, feature_cols[vx]] * 1.05,
                    centers[i, feature_cols[vy]] * 1.05,
                    str(i + 1), color='red', fontsize=8)
        if row == 0:
            ax.set_title(f'{vx.upper()} vs {vy.upper()}')
        if col == 0:
            ax.set_ylabel(region, fontsize=11)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)

fig.tight_layout()
plt.savefig('fig1_feature_space.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2 — Spatial distribution of clusters

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Cluster spatial distribution', fontsize=14)

for ax, region in zip(axes, region_names):
    r = regions[region]
    sc = ax.scatter(
        r['ds']['longitude'].values,
        r['ds']['latitude'].values,
        c=r['y_kmeans'], s=8, cmap='tab10', vmin=0, vmax=9, alpha=0.7)
    ax.set_title(region)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.colorbar(sc, ax=ax, label='Cluster', ticks=range(K))

fig.tight_layout()
plt.savefig('fig2_spatial.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 3 — Stacked bar: stc macro-group composition per cluster

Each panel is one region. Each bar is one cluster. Segments show the percentage of observations falling into each stc macro-group, averaged across all eight altitude levels.  
This is the key figure linking K-means regimes to the synergetic AC\_\_TC\_\_2B classification.

In [ ]:
stc_levels = ['stc_2500', 'stc_5000', 'stc_7500', 'stc_10000',
               'stc_12500', 'stc_15000', 'stc_17500', 'stc_20000']

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig.suptitle('stc macro-group composition per cluster (% averaged over all altitude levels)', fontsize=13)

group_names = list(STC_GROUPS.keys())

for ax, region in zip(axes, region_names):
    r = regions[region]
    # Average composition over all altitude levels
    comp_all = np.zeros((K, len(group_names)))
    for lev in stc_levels:
        comp, _ = stc_composition(r['ds_tc'], lev, r['y_kmeans'], K)
        comp_all += comp
    comp_all /= len(stc_levels)

    x_pos = np.arange(K)
    bottom = np.zeros(K)
    for j, group in enumerate(group_names):
        ax.bar(x_pos, comp_all[:, j], bottom=bottom,
               color=GROUP_COLORS[group], label=group if ax == axes[0] else '')
        bottom += comp_all[:, j]

    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'C{i+1}' for i in range(K)])
    ax.set_title(region)
    ax.set_xlabel('Cluster')

axes[0].set_ylabel('% composition')
axes[0].set_ylim(0, 100)
fig.legend(loc='lower center', ncol=len(group_names), bbox_to_anchor=(0.5, -0.12), fontsize=10)
fig.tight_layout()
plt.savefig('fig3_stc_composition.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 4 — Vertical profile of stc composition per cluster

For each region, shows how the stc macro-group composition changes with altitude for each cluster.  
Arranged as a 3 × K grid (one column per cluster, one row per region).

In [ ]:
# Altitude levels in km (for y-axis labelling)
altitudes = [2.5, 5.0, 7.5, 10.0, 12.5, 15.0, 17.5, 20.0]

fig, axes = plt.subplots(3, K, figsize=(4 * K, 12), sharey=True, sharex=True)
fig.suptitle('stc macro-group composition by altitude and cluster', fontsize=13, y=1.01)

for row, region in enumerate(region_names):
    r = regions[region]

    for cluster_idx in range(K):
        ax = axes[row, cluster_idx]

        # Build (n_levels × n_groups) composition matrix for this cluster
        comp_vertical = np.zeros((len(stc_levels), len(group_names)))
        for lev_i, lev in enumerate(stc_levels):
            comp, _ = stc_composition(r['ds_tc'], lev, r['y_kmeans'], K)
            comp_vertical[lev_i, :] = comp[cluster_idx, :]

        # Stacked horizontal bars (altitude on y-axis)
        left = np.zeros(len(stc_levels))
        for j, group in enumerate(group_names):
            ax.barh(altitudes, comp_vertical[:, j], left=left,
                    color=GROUP_COLORS[group], height=2.0,
                    label=group if (row == 0 and cluster_idx == 0) else '')
            left += comp_vertical[:, j]

        ax.set_xlim(0, 100)
        ax.set_yticks(altitudes)
        if cluster_idx == 0:
            ax.set_ylabel(f'{region}\nAltitude (km)', fontsize=9)
        if row == 0:
            ax.set_title(f'Cluster {cluster_idx + 1}', fontsize=11)
        if row == 2:
            ax.set_xlabel('%')

fig.legend(loc='lower center', ncol=len(group_names), bbox_to_anchor=(0.5, -0.06), fontsize=10)
fig.tight_layout()
plt.savefig('fig4_vertical_profiles.png', dpi=150, bbox_inches='tight')
plt.show()